In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

dataset = load_breast_cancer()

X = dataset.data
y = dataset.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()

X_train_scaler = scaler.fit_transform(X_train)
X_test_scaler = scaler.transform(X_test)

X_train = torch.tensor(X_train_scaler, dtype=torch.float32)
X_test = torch.tensor(X_test_scaler, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_test = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

train_dataset = TensorDataset(X_train, y_train)

test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)


class BinaryClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(30, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.model(x)

model = BinaryClassifier()
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
epochs = 100

for epoch in range(epochs):
    model.train()
    total_loss = 0i
    for X_batch, y_batch in train_loader:
        output = model(X_batch)
        loss = criterion(output, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch [{epoch + 1}/{epochs}] "
            f"Loss: {total_loss / len(train_loader):.4f}"
        )
model.eval()

with torch.no_grad():
    output = model(X_test)
    probability = torch.sigmoid(output)
    prediction = (probability >= 0.5).float()
    accuracy = (prediction == y_test).float().mean()

print()
print("Test Accuracy :", accuracy.item())


Epoch [10/100] Loss: 0.0562
Epoch [20/100] Loss: 0.0390
Epoch [30/100] Loss: 0.0186
Epoch [40/100] Loss: 0.0115
Epoch [50/100] Loss: 0.0065
Epoch [60/100] Loss: 0.0038
Epoch [70/100] Loss: 0.0023
Epoch [80/100] Loss: 0.0015
Epoch [90/100] Loss: 0.0011
Epoch [100/100] Loss: 0.0008

Test Accuracy : 0.9473684430122375
